In [1]:
import numpy as np
from numba import cuda
import math

# Define block size
BLOCK_SIZE = 256

@cuda.jit
def array_sum_kernel(d_input, d_output):
    # Allocate shared memory dynamically shared across the thread block
    s_data = cuda.shared.array(shape=BLOCK_SIZE, dtype=cuda.float32)

    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x

    # Calculate unique global index for this thread
    global_idx = bx * bdim + tx

    # Step 1: Load data from global memory into Shared Memory
    if global_idx < d_input.size:
        s_data[tx] = d_input[global_idx]
    else:
        s_data[tx] = 0.0  # Pad with 0 if out of array bounds

    # Synchronize to make sure all threads have finished loading shared memory
    cuda.syncthreads()

    # Step 2: Perform reduction in shared memory (Tree-based reduction)
    # Start with stride = half block size, and divide by 2 each iteration
    stride = bdim // 2
    while stride > 0:
        if tx < stride:
            s_data[tx] += s_data[tx + stride]
        cuda.syncthreads()  # Ensure all partial sums are done before next stride
        stride //= 2

    # Step 3: Write the block's total sum to global memory
    # Only thread 0 of each block does this
    if tx == 0:
        d_output[bx] = s_data[0]

def parallel_sum(h_input):
    n = h_input.size

    # Configure grid layout
    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(n / threads_per_block)

    # Allocate memory on the GPU device
    d_input = cuda.to_device(h_input.astype(np.float32))
    # Each block outputs its own partial sum
    d_output = cuda.device_array(blocks_per_grid, dtype=np.float32)

    # Launch the kernel
    array_sum_kernel[blocks_per_grid, threads_per_block](d_input, d_output)

    # Copy the partial block sums back to the host
    h_partial_sums = d_output.copy_to_host()

    # Sum the few remaining block totals on the CPU
    final_sum = np.sum(h_partial_sums)
    return final_sum

# -------------------------------------------------------------------
# Host Execution Verification
# -------------------------------------------------------------------
if __name__ == "__main__":
    # Create an array of 1,000,000 elements (e.g., all ones for easy testing)
    array_size = 1_000_000
    test_array = np.ones(array_size, dtype=np.float32)

    print(f"Array size: {array_size:,} elements.")

    # Run GPU parallel sum
    gpu_result = parallel_sum(test_array)

    # Run NumPy CPU baseline sum
    cpu_result = np.sum(test_array)

    # Verify correctness
    print(f"GPU Calculated Sum: {gpu_result}")
    print(f"CPU Calculated Sum: {cpu_result}")

    if np.isclose(gpu_result, cpu_result):
        print("Success! The GPU sum matches the CPU sum.")
    else:
        print("Mismatch! Something went wrong.")

Array size: 1,000,000 elements.
GPU Calculated Sum: 1000000.0
CPU Calculated Sum: 1000000.0
Success! The GPU sum matches the CPU sum.
